In [3]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

folder_path = r"D:\OneDrive\Trading\Market Making\data\runs\run_20260607_062400"

snapshots = pd.read_parquet(os.path.join(folder_path, "snapshots.parquet"))

snapshots

,ts,symbol,best_bid,best_ask,mid,microprice,best_bid_tick,best_ask_tick,mid_tick,spread,...,my_ask,my_bid_tick,my_ask_tick,bid_distance_touch,ask_distance_touch,bid_distance_spread,ask_distance_spread,bid_delta,ask_delta,quote_churn
0,1780713724914,BTCUSDT,60980.06,60980.07,60980.065,60980.060331,6098006,6098007,6098006,0.01,...,60980.07,6097956,6098007,-0.50,0.00,-0.51,0.01,0.0,0.0,0.0
1,1780713725014,BTCUSDT,60980.06,60980.07,60980.065,60980.060331,6098006,6098007,6098006,0.01,...,60980.07,6097957,6098007,-0.49,0.00,-0.50,0.01,0.0,0.0,0.0
2,1780713725114,BTCUSDT,60980.06,60980.07,60980.065,60980.060338,6098006,6098007,6098006,0.01,...,60980.07,6097957,6098007,-0.49,0.00,-0.50,0.01,0.0,0.0,0.0
3,1780713725214,BTCUSDT,60980.06,60980.07,60980.065,60980.060374,6098006,6098007,6098006,0.01,...,60980.07,6097957,6098007,-0.49,0.00,-0.50,0.01,0.0,0.0,0.0
4,1780713725314,BTCUSDT,60980.06,60980.07,60980.065,60980.060372,6098006,6098007,6098006,0.01,...,60980.07,6097957,6098007,-0.49,0.00,-0.50,0.01,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223,1780713747214,BTCUSDT,60960.70,60960.71,60960.705,60960.705438,6096070,6096071,6096070,0.01,...,60961.14,6096070,6096114,0.00,0.43,-0.01,0.44,0.0,0.0,0.0
224,1780713747314,BTCUSDT,60960.70,60960.71,60960.705,60960.705461,6096070,6096071,6096070,0.01,...,60960.92,6096070,6096092,0.00,0.21,-0.01,0.22,0.0,0.0,0.0
225,1780713747414,BTCUSDT,60960.70,60960.71,60960.705,60960.705461,6096070,6096071,6096070,0.01,...,60960.90,6096070,6096090,0.00,0.19,-0.01,0.20,0.0,0.0,0.0
226,1780713747514,BTCUSDT,60960.70,60960.71,60960.705,60960.705461,6096070,6096071,6096070,0.01,...,60960.89,6096070,6096089,0.00,0.18,-0.01,0.19,0.0,0.0,0.0


In [ ]:
"""
Key Research Questions

This project is designed to investigate:

Which regimes favor passive liquidity provision? medium frequency trends (1000ms rolling window)

"""

# STEP 1 — Load raw data
df = snapshots
df["ts"] = pd.to_datetime(df["ts"], unit="ms")
df = df.set_index("ts")

# STEP 2 — Build REGIME FEATURES (ONLY past info) slower trends - 1000ms

"""
2. Choose regime window (critical design choice)
Start simple:
"""

feature_cols = [
    "volatility",
    "spread",
    "order_imbalance",
    "trade_imbalance",
    "quote_churn",
    "inventory",
    "inventory_vol",
    "microprice_error"
]

WINDOW = "1s"   # later try 2s, 5s

regime_df = pd.DataFrame()

regime_df["volatility"] = df["mid"].pct_change().rolling(WINDOW).std()
regime_df["spread"] = df["spread"].rolling(WINDOW).mean()
regime_df["order_imbalance"] = df["order_imbalance"].rolling(WINDOW).mean()
regime_df["trade_imbalance"] = df["trade_imbalance"].rolling(WINDOW).mean()
regime_df["quote_churn"] = df["quote_churn"].rolling(WINDOW).mean()
regime_df["inventory"] = df["inventory"].rolling(WINDOW).mean()
regime_df["inventory_vol"] = df["inventory"].rolling(WINDOW).std()
regime_df["microprice_error"] = (df["mid"] - df["microprice"]).rolling(WINDOW).mean()

regime_df = regime_df.dropna()
regime_df

In [ ]:
# STEP 3 — Train regime model
scaler = StandardScaler()

X = regime_df[feature_cols].values
X_scaled = scaler.fit_transform(X)

n_regimes = 3  # start small: 2–5 max

model = GaussianMixture(
    n_components=n_regimes,
    covariance_type="full",
    random_state=42
)

regime_df["regime"] = model.fit_predict(X_scaled)

In [ ]:
eval_df = df.copy()

times = eval_df.index          # DatetimeIndex
mid = eval_df["mid"].values

HORIZON = pd.Timedelta(milliseconds=1000)

future_return = np.full(len(df), np.nan)
future_volatility = np.full(len(df), np.nan)
future_direction = np.full(len(df), np.nan)

for i in range(len(df)):

    target_time = times[i] + HORIZON

    # first observation at or after t + 1000ms
    j = times.searchsorted(target_time)

    if j >= len(df):
        continue

    p0 = mid[i]
    p1 = mid[j]

    # future window [i, j]
    window = mid[i:j+1]

    # Need at least 2 observations
    if len(window) < 2:
        continue

    # 1. Future return
    future_return[i] = (p1 - p0) / p0

    # 2. Realized volatility over next 1000ms
    returns = np.diff(window) / window[:-1]
    future_volatility[i] = np.std(returns)

    # 3. Future direction
    # If result ≈ +1
    # almost always up moves after this regime
    # strong bullish bias
    # If result ≈ -1
    # almost always down moves after this regime
    # bearish bias
    # If result ≈ 0
    # no directional bias
    # pure noise / mean reversion / stable
    future_direction[i] = np.sign(p1 - p0)

eval_df["future_return"] = future_return
eval_df["future_volatility"] = future_volatility
eval_df["future_direction"] = future_direction

eval_df = eval_df.dropna(subset=[ "future_return", "future_volatility", "future_direction"])
eval_df

,symbol,best_bid,best_ask,mid,microprice,best_bid_tick,best_ask_tick,mid_tick,spread,order_imbalance,...,future_return_100ms,future_mid_500ms,future_return_500ms,future_mid_1000ms,future_return_1000ms,future_mid_5000ms,future_return_5000ms,future_return,future_volatility,future_direction
ts,,,,,,,,,,,,,,,,,,,,,
2026-06-06 02:42:04.914,BTCUSDT,60980.06,60980.07,60980.065,60980.060331,6098006,6098007,6098006,0.01,-0.931144,...,0.0,60980.065,0.0,60980.065,0.0,60964.005,-0.000263,0.0,0.0,0.0
2026-06-06 02:42:05.014,BTCUSDT,60980.06,60980.07,60980.065,60980.060331,6098006,6098007,6098006,0.01,-0.931144,...,0.0,60980.065,0.0,60980.065,0.0,60964.005,-0.000263,0.0,0.0,0.0
2026-06-06 02:42:05.114,BTCUSDT,60980.06,60980.07,60980.065,60980.060338,6098006,6098007,6098006,0.01,-0.929655,...,0.0,60980.065,0.0,60980.065,0.0,60964.005,-0.000263,0.0,0.0,0.0
2026-06-06 02:42:05.214,BTCUSDT,60980.06,60980.07,60980.065,60980.060374,6098006,6098007,6098006,0.01,-0.922498,...,0.0,60980.065,0.0,60980.065,0.0,60964.005,-0.000263,0.0,0.0,0.0
2026-06-06 02:42:05.314,BTCUSDT,60980.06,60980.07,60980.065,60980.060372,6098006,6098007,6098006,0.01,-0.922938,...,0.0,60980.065,0.0,60980.065,0.0,60964.005,-0.000263,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-06 03:16:15.314,BTCUSDT,60883.43,60883.44,60883.435,60883.431172,6088343,6088344,6088344,0.01,-0.762572,...,0.0,60883.435,0.0,60883.435,0.0,NaN,NaN,0.0,0.0,0.0
2026-06-06 03:16:15.414,BTCUSDT,60883.43,60883.44,60883.435,60883.431169,6088343,6088344,6088344,0.01,-0.763103,...,0.0,60883.435,0.0,60883.435,0.0,NaN,NaN,0.0,0.0,0.0
2026-06-06 03:16:15.514,BTCUSDT,60883.43,60883.44,60883.435,60883.430585,6088343,6088344,6088344,0.01,-0.879783,...,0.0,60883.435,0.0,60883.435,0.0,NaN,NaN,0.0,0.0,0.0


In [ ]:
# STEP 5 — ALIGN BOTH DATASETS

# This is the missing step in your code.

# Now regime + outcome are aligned.

final = regime_df.merge(
    eval_df[["future_return", "future_volatility", "future_direction"]],
    left_index=True,
    right_index=True,
    how="inner"
)

# STEP 6 — ANALYZE REGIMES
regime_outcomes = final.groupby("regime").agg({
    "future_return": "mean",
    "future_volatility": "mean",
    "future_direction": "mean"
})

z = final.copy()

for col in feature_cols:
    z[col] = (z[col] - z[col].mean()) / z[col].std()

regime_profile = (
    z.groupby("regime")[feature_cols]
    .mean()
    .round(2)
)

full_profile = pd.DataFrame(regime_profile.join(regime_outcomes))
full_profile

,volatility,spread,order_imbalance,trade_imbalance,quote_churn,inventory,inventory_vol,microprice_error,future_return,future_volatility,future_direction
regime,,,,,,,,,,,
0,1.93,2.11,0.00,-0.15,NaN,0.02,1.44,-0.10,0.000012,0.000010,0.043785
1,-0.05,-0.08,0.05,-0.00,NaN,0.69,-0.05,-0.03,-0.000003,0.000009,-0.058434
2,-0.11,-0.08,-0.10,0.02,NaN,-1.31,-0.05,0.08,0.000003,0.000008,-0.072946


In [ ]:
regime_labels = {
    0: "trending",
    1: "toxic",
    2: "low_vol",
}

artifact = {
    "scaler": scaler,
    "model": model,
    "feature_cols": feature_cols,
    "n_regimes": n_regimes,
    "window": WINDOW,
    "horizon_ms": 1000,
    "regime_labels": regime_labels
}

joblib.dump(artifact, "data/regime_model_4.pkl")

['regime_model_4.pkl']